# Ion channel fitting of a hyperpolarization-activated (HCN) channel

Fits an HCN ion channel model from activation recordings, and registers it. Only the Activation protocol of the recordings is used. The reversal potential is not measured by that protocol, so it is set in the model block (default -30 mV).

Running the cells that generate and run the campaign registers a model in the project.

In [ ]:
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask
from obi_one import LAB_ID_STAGING_TEST, PROJECT_ID_STAGING_TEST
from obi_one.core.info import Info
from obi_one.scientific.from_id.ion_channel_recording_from_id import IonChannelRecordingFromID
from obi_one.scientific.tasks.ion_channel_modeling.config import (
    HyperpolarizationActivatedIonChannelModel,
    IonChannelFittingScanConfig,
)

# The project that holds the HCN recordings. Recordings are private to their project until the
# data is made public, so use a project where you can read them.
virtual_lab_id = LAB_ID_STAGING_TEST
project_id = PROJECT_ID_STAGING_TEST

from entitysdk import Client, ProjectContext
from obi_auth import get_token

obi_one_api_url = "http://127.0.0.1:8100"
token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
db_client = Client(api_url="https://staging.openbraininstitute.org/api/entitycore", project_context=project_context, token_manager=token)

# IonChannelRecording IDs

In [ ]:
# IDs of HCN IonChannelRecordings of your project, fitted together into one model.
# They must share the same temperature. Find them in the platform and paste them here.
recording_ids = ["<hcn-recording-id>"]

# Create Ion Channel Fitting Form using HCN Ion Channel Recording(s)

In [ ]:
icm_conf = IonChannelFittingScanConfig(
    initialize=IonChannelFittingScanConfig.Initialize(
        recordings=tuple(IonChannelRecordingFromID(id_str=id_) for id_ in recording_ids),
        # model_properties (ion channel name, temperature, ...) are left unset: they are taken
        # from the recordings
    ),
    info=Info(
        campaign_name="HCN Ion Channel Modeling Example",
        campaign_description="HCN channel fitted from activation recordings.",
    ),
    model_type=HyperpolarizationActivatedIonChannelModel(
        mtau_eq="bell_fit_mtau",  # or "thermo_fit_mtau"
        m_power=1,
        reversal_potential=-30.0,  # mV. Can be a list, to scan over it
    ),
)

# Validated Config
validated_icm_conf = icm_conf.validated_config()

# Generate GridScanGenerationTask (locally) and run it. This registers a model in the project.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    grid_scan = GridScanGenerationTask(form=validated_icm_conf, coordinate_directory_option="ZERO_INDEX", output_root=temp_dir)
    grid_scan.multiple_value_parameters(display=True)
    grid_scan.coordinate_parameters(display=True)
    grid_scan.execute(db_client=db_client)
    run_tasks_for_generated_scan(grid_scan, db_client=db_client)

# Test endpoint. Service must be launched first (make run-local)

In [ ]:
import requests

# Construct the full endpoint URL
url = f"{obi_one_api_url}/generated/ion-channel-fitting-scan-config-generate-grid"

# Prepare headers
headers = {
    "Authorization": f"Bearer {token}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}
if virtual_lab_id:
    headers["virtual-lab-id"] = virtual_lab_id
if project_id:
    headers["project-id"] = project_id

# Construct request
request_body = {
    "type": "IonChannelFittingScanConfig",
    "info": {
        "type": "Info",
        "campaign_name": "HCN Ion Channel Modeling Example",
        "campaign_description": "HCN channel fitted from activation recordings.",
    },
    "initialize": {
        "recordings": [
            {"type": "IonChannelRecordingFromID", "id_str": id_} for id_ in recording_ids
        ],
    },
    "model_type": {
        "type": "HyperpolarizationActivatedIonChannelModel",
        "minf_eq": "sig_fit_minf",
        "mtau_eq": "bell_fit_mtau",
        "m_power": 1,
        "reversal_potential": -30.0,
    },
}

response = requests.post(url, headers=headers, json=request_body)
if response.status_code == 200:
    data = response.json()
    print("Success:", data)
else:
    print(f"Error {response.status_code}: {response.text}")